# 1/3 — GNN4ID (bản chạy được trên Linux, input = 2 file pcap)

Copy của `GNN4ID.ipynb`, giữ nguyên luồng của tác giả, chỉ sửa path + vài chỗ Windows-only.
Input: `data/Debug and Trace/XSS.pcap` và `DictionaryBruteForce.pcap`.

**Thứ tự chạy 3 notebook:**
1. Notebook này, phần A (pcap → csv flow+packet → thêm feature rolling)
2. `2_Data_preprocessing_pcap.ipynb` (split train/test → gộp class → 1 file train + 1 file test)
3. Quay lại notebook này, phần B (csv → graph objects `.pt`)
4. `3_GNN4ID_Model_pcap.ipynb` (train + test HGNN)

Chọn kernel = env **xmfgnn** trước khi chạy.

## Phần A

In [ ]:
from Utility.Functions import *
from Utility.Additional_Features import *
import glob
import os
import shutil
import subprocess
import sys
from tqdm import tqdm

In [ ]:
# [auto-break] Đổi AUTO_BREAK = True nếu muốn dừng lại ở từng hàm của tác giả.
#   - Bấm "Debug Cell"  -> debugger dừng THẬT bên trong hàm (F5 = hàm kế, F10 = từng dòng)
#   - Bấm "Run Cell"    -> VS Code hiện ô nhập ở trên cùng cửa sổ, phải gõ Enter mới chạy tiếp
#                          (s = bỏ qua hàm đó, q = tắt hẳn auto-break)
# Chạy cell này TRƯỚC các cell khác.
AUTO_BREAK = False          # True = bật

if AUTO_BREAK:
    from Debug.autobreak import install
    installed, mode = install(
        stages=["split", "features", "combine", "graphs", "model"],
        max_hits=0,          # 0 = dừng ở MỌI lần gọi; 1 = chỉ lần gọi đầu của mỗi hàm
        pause="input",       # chờ Enter khi chạy bằng Run Cell (lúc không có debugger)
    )
    print("auto-break [%s]: %d hàm" % (mode, len(installed)))
    for x in installed:
        print("  ", x)


In [ ]:
# [linux] Toàn bộ path của notebook gốc là Windows (F:/CIC_IOT/...) -> đổi sang path máy này.
# 2 file pcap gốc KHÔNG bị đụng tới: mọi thứ sinh ra nằm trong .../Debug and Trace/nb_run/
import os

BASE          = "/home/tutay/Tutay/Tutay_Sec/XG_NID"
PCAP_SRC      = os.path.join(BASE, "data", "Debug and Trace")              # 2 file pcap input
WORK          = os.path.join(PCAP_SRC, "nb_run")                           # thư mục làm việc
Out_Directory = os.path.join(WORK, "Packet_Level_Data")                    # pcap sau khi đổi tên
Out_path      = os.path.join(WORK, "Extracted_Flow_Features") + os.sep     # csv + graph objects
print("WORK     =", WORK)
print("Out_path =", Out_path)

### Bỏ bước giải nén .tar.gz

Notebook gốc giải nén các file `.tar.gz` của CIC-IoT2023. Ở đây đã có sẵn 2 file pcap nên chỉ
copy chúng vào thư mục làm việc (copy để `rename_files` và bước xoá pcap sau khi trích xuất
không đụng vào file gốc).

In [ ]:
pcap_dir = os.path.join(Out_Directory, "pcap")   # rename_files glob theo kiểu <dir>/*/*pcap
os.makedirs(pcap_dir, exist_ok=True)
os.makedirs(Out_path, exist_ok=True)

for f in ["XSS.pcap", "DictionaryBruteForce.pcap"]:
    shutil.copy2(os.path.join(PCAP_SRC, f), os.path.join(pcap_dir, f))

print(os.listdir(pcap_dir))

### Đổi tên file PCAP

Tên file quyết định nhãn class về sau (`split_csv` / `Combining_classes` lấy phần trước dấu `-`).
`XSS` → `WebBased-XSS`, `DictionaryBruteForce` → `BruteForce-Dictionary`.

In [ ]:
name_mapping = {'Benign': 'Benign-Benign' ,
          'DDoS-ACK_Fragmentation':'DDos-AckFrg',
          'DDoS-UDP_Flood':'DDos-UDPFlood',
         'DDos-SlowLoris':'DDos-SlowLoris',
         'DDoS-ICMP_Flood':'DDos-ICMPFlood',
         'DDoS-RSTFINFlood' :'DDos-RSTFIN',
         'DDoS-PSHACK_Flood':'DDos-PSHACK',
         'DDoS-HTTP_Flood':'DDos-HTTPFlood',
         'DDoS-UDP_Fragmentation':'DDos-UDPFrg' ,
         'DDoS-ICMP_Fragmentation':'DDos-ICMPFrg',
         'DDoS-TCP_Flood':'DDos-TCPFlood',
         'DDoS-SYN_Flood':'DDos-SYNFlood',
         'DDoS-SynonymousIP_Flood':'DDos-SynonymousIPFlood' ,
          'DoS-TCP_Flood':'Dos-TCPFlood',
          'DoS-HTTP_Flood':'Dos-HTTPFlood',
          'DoS-SYN_Flood':'Dos-SYNFlood',
          'DoS-UDP_Flood':'Dos-UDPFlood',
          'Recon-PingSweep':'Recon-PingSweep',
          'Recon-OSScan':'Recon-OSScan',
          'VulnerabilityScan':'Recon-VulScan',
          'Recon-PortScan':'Recon-PortScan',
          'Recon-HostDiscovery':'Recon-HostDisc',
          'SqlInjection':'WebBased-SqlInject',
          'CommandInjection':'WebBased-CmmdInject',
          'Backdoor_Malware':'WebBased-BckdoorMalware',
          'Uploading_Attack':'WebBased-UploadAttack',
          'XSS':'WebBased-XSS',
          'BrowserHijacking':'Webbased-BrwserHijack',
          'DictionaryBruteForce':'BruteForce-Dictionary',
          'MITM-ArpSpoofing':'Spoofing-ARP',
          'DNS_Spoofing':'Spoofing-DNS',
          'Mirai-greip_flood':'Mirai-GREIP',
          'Mirai-greeth_flood':'Mirai-Greeth',
          'Mirai-udpplain':'Mirai-UDPPlain'
         }

In [ ]:
## Function that rename the files
rename_files(Out_Directory, name_mapping)

### Trích xuất feature từ PCAP

Chạy `Utility/Feature_extractor_flow_packet_combined.py` (NFStreamer + plugin `My_Custom`)
cho từng pcap → 1 file csv/pcap, mỗi dòng = 1 flow, kèm thông tin từng packet (payload,
delta_time, direction, ip_size, flags...).

[linux] `'python'` → `sys.executable` để chạy đúng python của env đang mở notebook.

In [ ]:
directory = os.path.join(Out_Directory, "**", "*pcap")
List_of_PCAP = glob.glob(directory)
print(List_of_PCAP)

# [guard] Cell rename_files ở trên PHẢI chạy trước cell này. Nếu quên, tên csv sẽ là
# XSS.csv thay vì WebBased-XSS_0.csv, và Combining_classes ở notebook 2 sẽ báo
# "ValueError: No objects to concatenate" vì không tìm thấy file bắt đầu bằng tên class.
assert List_of_PCAP and all("-" in os.path.basename(p) for p in List_of_PCAP), (
    "Chưa chạy cell rename_files! Tên pcap phải dạng 'WebBased-XSS_0.pcap'. Đang có: %s"
    % [os.path.basename(p) for p in List_of_PCAP])

feature_Extractor = 'Utility/Feature_extractor_flow_packet_combined.py'

for single_pcap_file in tqdm(List_of_PCAP):
    print("Reading File: ", os.path.basename(single_pcap_file))
    completed_process = subprocess.run([sys.executable, feature_Extractor, single_pcap_file, Out_path],
                                       capture_output=True)
    if completed_process.returncode != 0:
        print(completed_process.stderr.decode()[-2000:])
    os.remove(single_pcap_file)  # xoá pcap đã copy (file gốc vẫn còn trong Debug and Trace/)
    print("**Extraction Completed For: ", os.path.basename(single_pcap_file), "**")

print(os.listdir(Out_path))

### Thêm feature theo thời gian (Additional / Explainable Features)

`additional_features()` sắp xếp flow theo `bidirectional_first_seen_ms` rồi tính 16 feature
rolling-window theo destination (Rolling_UDP_Sum, Rolling_SYN_Sum, ...), đồng thời one-hot
`expiration_id` + `protocol`. Ghi đè lên chính file csv.

In [ ]:
Extracted_Features_Files = glob.glob(os.path.join(Out_path, "*csv"))
for file in tqdm(Extracted_Features_Files):
    additional_features(file)

import pandas as pd
pd.read_csv(Extracted_Features_Files[0]).shape

---
# ⛔ DỪNG Ở ĐÂY

Chạy `2_Data_preprocessing_pcap.ipynb` cho xong rồi quay lại chạy tiếp phần B bên dưới.

---
## Phần B — chuyển csv thành graph objects

Mỗi flow → 1 heterogeneous graph: **flow node** (feature thống kê) + **packet node**
(1500 byte payload / packet), nối bằng **contain edge** (flow → từng packet, attr =
direction/ip_size/transport_size/payload_size) và **link edge** (packet i → packet i+1,
attr = delta_time).

In [ ]:
## Dictionary for classifying Classes and Assigning them Class number for reference
Dict_x = {'Benign': 0 ,
          'WebBased': 1,
          'Spoofing': 2,
          'Recon' : 3,
          'Mirai' : 4,
          'Dos' : 5,
          'DDos' : 6,
          'BruteForce': 7
         }

dir = Out_path                                              # nơi lưu graph objects (-> processed/)
Files = glob.glob(os.path.join(Out_path, "train", "*.csv"))  # file train gộp từ notebook 2
print(Files)

In [ ]:
## Generation of graph data obejcts (TRAIN)
data_Hetero = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files,
                          skip_processing=False, test=False, single_file=True)
print("train graphs:", len(data_Hetero))
data_Hetero[0]

In [ ]:
## Generation of graph data obejcts (TEST)
Files_test = glob.glob(os.path.join(Out_path, "test", "df_class_8_test.csv"))
print(Files_test)
data_Hetero_test = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files_test,
                               skip_processing=False, test=True, single_file=True)
print("test graphs:", len(data_Hetero_test))

Xong → sang `3_GNN4ID_Model_pcap.ipynb` để train/test model.